## Device & Environment Specifications

| Specification | Details |
|---|---|
| **Platform** | Google Colaboratory |
| **Accelerator** | NVIDIA Tesla T4 |
| **GPU Memory** | 15 GB GDDR6 |
| **GPU Architecture** | Turing (SM75), 2560 CUDA cores |
| **FP32 Peak** | ~8.1 TFLOPS |
| **INT8 Peak** | ~130 TOPS (Tensor Cores) |
| **CPU** | Intel Xeon @ ~2.3 GHz (2 vCPUs) |
| **System RAM** | ~12 GB |
| **CUDA Version** | 12.x |
| **PyTorch** | ≥ 2.0 |
| **Transformers** | ≥ 4.35 |
| **Mixed Precision** | AMP (torch.cuda.amp) |
| **Pipeline** | Pretrained → Fine-tune (full dataset, cosine LR, grad accum) → DuQuant (Rot.pkl + activation calibration + FFN-only + LWC + QAT) → Benchmark |


In [ ]:
!pip install -q -U transformers datasets evaluate accelerate pynvml scipy
!pip install -q "scikit-learn>=1.2,<1.9"

import os, pickle, math, torch
import time, gc, warnings, threading, tempfile, copy
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup,
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    import pynvml
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    print('GPU Name    :', pynvml.nvmlDeviceGetName(handle))
    print('GPU Memory  :', round(pynvml.nvmlDeviceGetMemoryInfo(handle).total / 1e9, 2), 'GB')
    print('CUDA Version:', torch.version.cuda)
print('PyTorch     :', torch.__version__)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True


In [ ]:
def _build_rot(n):
    R = torch.eye(n, dtype=torch.float32)
    for i in range(n - 1):
        cos = 1.0 / math.sqrt(i + 2)
        sin = math.sqrt(i + 1) / math.sqrt(i + 2)
        rot_i = torch.eye(n, dtype=torch.float32)
        rot_i[i, i] = cos;   rot_i[i+1, i+1] = cos
        rot_i[i, i+1] = -sin; rot_i[i+1, i] = sin
        R = rot_i @ R
    R[[0, n-1]] = R[[n-1, 0]]
    R[:, [0, n-1]] = R[:, [n-1, 0]]
    return R

ROT_PKL_PATH = 'Rot_bert.pkl'
if not os.path.exists(ROT_PKL_PATH):
    print('Building Rot_bert.pkl...')
    sizes = [2,4,8,16,32,48,64,78,96,104,128,192,256,312,384,512,768]
    dic   = {n: _build_rot(n) for n in sizes}
    pickle.dump(dic, open(ROT_PKL_PATH, 'wb'))
    print(f'Built with keys: {list(dic.keys())}')
else:
    print(f'Rot_bert.pkl found.')

_ROT_PKL = pickle.load(open(ROT_PKL_PATH, 'rb'))
print('Loaded rotation cache, keys:', list(_ROT_PKL.keys())[:8])


In [ ]:
MODELS = {
    'TinyBERT':   'huawei-noah/TinyBERT_General_4L_312D',
    'DistilBERT': 'distilbert-base-uncased',
    'AlBERT':     'albert-base-v2',
    'MobileBERT': 'google/mobilebert-uncased',
    'BERT-base':  'bert-base-uncased',
    'GPT-2':      'gpt2',
}

GPT2_MODELS = {'GPT-2'}

DATASETS = {
    'SST2': ('stanfordnlp/sst2',  None,   'train', 'validation',         'sentence',               'label'),
    'QNLI': ('nyu-mll/glue',      'qnli', 'train', 'validation',         ('question','sentence'),   'label'),
    'MNLI': ('nyu-mll/glue',      'mnli', 'train', 'validation_matched', ('premise','hypothesis'),  'label'),
    'QQP':  ('nyu-mll/glue',      'qqp',  'train', 'validation',         ('question1','question2'), 'label'),
    'RTE':  ('nyu-mll/glue',      'rte',  'train', 'validation',         ('sentence1','sentence2'), 'label'),
    'CoLA': ('nyu-mll/glue',      'cola', 'train', 'validation',         'sentence',               'label'),
    'MRPC': ('nyu-mll/glue',      'mrpc', 'train', 'validation',         ('sentence1','sentence2'), 'label'),
    'STSB': ('nyu-mll/glue',      'stsb', 'train', 'validation',         ('sentence1','sentence2'), 'label'),
    'WNLI': ('nyu-mll/glue',      'wnli', 'train', 'validation',         ('sentence1','sentence2'), 'label'),
}

NUM_LABELS = {
    'SST2': 2, 'QNLI': 2, 'MNLI': 3, 'QQP': 2,
    'RTE': 2, 'CoLA': 2, 'MRPC': 2, 'STSB': 1, 'WNLI': 2,
}

BATCH_SIZE        = 8
FINETUNE_BATCH    = 4
GRAD_ACCUM_STEPS  = 4
MAX_SAMPLES       = 300
FINETUNE_SAMPLES  = None
MAX_LENGTH        = 128
FINETUNE_EPOCHS   = 3
QAT_EPOCHS        = 1
LR                = 2e-5
WARMUP_RATIO      = 0.1
POLL_INTERVAL_S   = 0.01
PERMUTATION_TIMES = 1
MAX_ROTATION_STEP = 256
BLOCK_SIZE_DEFAULT= 128
N_CALIB_SAMPLES   = 32
SWC_RATIO         = 0.90
LAC_RATIO         = 0.90
LWC_LR            = 1e-4
LWC_EPOCHS        = 5
CLIPMIN           = 1e-5
CLIPMAX           = 1e4

from torch.cuda.amp import autocast, GradScaler
SCALER = GradScaler(enabled=torch.cuda.is_available())
print('Config ready. Effective fine-tune batch:', FINETUNE_BATCH * GRAD_ACCUM_STEPS)


In [ ]:
NVML_AVAILABLE = False
try:
    import pynvml as _pynvml
    _pynvml.nvmlInit()
    _nvml_handle = _pynvml.nvmlDeviceGetHandleByIndex(0)
    NVML_AVAILABLE = True
except Exception:
    pass

class PowerSampler:
    def __init__(self):
        self._samples = []; self._running = False; self._thread = None
    def _poll(self):
        while self._running:
            if NVML_AVAILABLE:
                try: self._samples.append(_pynvml.nvmlDeviceGetPowerUsage(_nvml_handle))
                except: pass
            time.sleep(POLL_INTERVAL_S)
    def start(self):
        self._samples = []; self._running = True
        self._thread = threading.Thread(target=self._poll, daemon=True); self._thread.start()
    def stop(self):
        self._running = False; self._thread.join(timeout=0.5)
        return float(np.mean(self._samples)) if self._samples else 0.0

def get_texts(batch, text_col):
    if isinstance(text_col, tuple):
        return list(zip(batch[text_col[0]], batch[text_col[1]]))
    return batch[text_col]

def tokenize(tok, texts, labels=None, is_regression=False):
    if isinstance(texts[0], tuple):
        enc = tok([t[0] for t in texts], [t[1] for t in texts],
                  truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors='pt')
    else:
        enc = tok(list(texts), truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors='pt')
    if labels is not None:
        dtype = torch.float if is_regression else torch.long
        enc['labels'] = torch.tensor(labels, dtype=dtype)
    return enc

def memory_mb(model):
    with tempfile.NamedTemporaryFile(delete=False, suffix='.pt') as f:
        torch.save(model.state_dict(), f.name)
        size_mb = os.path.getsize(f.name) / (1024 * 1024)
    os.remove(f.name)
    return round(size_mb, 2)

print('Utilities ready.')


In [ ]:
import torch.nn as nn
try:
    from transformers.pytorch_utils import Conv1D as HF_Conv1D
    HAS_CONV1D = True
except ImportError:
    HAS_CONV1D = False

def patch_gpt2_conv1d(model):
    """Replace HuggingFace Conv1D with equivalent nn.Linear for DuQuant compatibility."""
    if not HAS_CONV1D:
        return model
    for name, module in list(model.named_modules()):
        if isinstance(module, HF_Conv1D):
            linear = nn.Linear(module.weight.shape[0], module.weight.shape[1], bias=module.bias is not None)
            linear.weight = nn.Parameter(module.weight.T.contiguous())
            if module.bias is not None:
                linear.bias = nn.Parameter(module.bias.clone())
            parts = name.split('.')
            parent = model
            for p in parts[:-1]: parent = getattr(parent, p)
            setattr(parent, parts[-1], linear)
    return model

def setup_gpt2_tokenizer(tok):
    tok.pad_token    = tok.eos_token
    tok.padding_side = 'left'
    return tok

print("GPT-2 helpers ready. Conv1D patching:", "enabled" if HAS_CONV1D else "fallback mode")


In [ ]:
def finetune(model, tok, ds_cfg, n_labels, ds_name, epochs=None, lr=None, desc='Fine-tuning'):
    path, config, train_split, _, text_col, label_col = ds_cfg
    is_regression = (ds_name == 'STSB')
    _epochs = epochs or FINETUNE_EPOCHS
    _lr     = lr or LR

    ds = load_dataset(path, config, split=train_split) if config else load_dataset(path, split=train_split)
    n  = len(ds) if FINETUNE_SAMPLES is None else min(FINETUNE_SAMPLES, len(ds))
    ds = ds.shuffle(seed=42).select(range(n))

    model.to(DEVICE).train()
    optimizer    = torch.optim.AdamW(model.parameters(), lr=_lr, weight_decay=0.01)
    total_steps  = (n // FINETUNE_BATCH // GRAD_ACCUM_STEPS + 1) * _epochs
    warmup_steps = max(1, int(total_steps * WARMUP_RATIO))
    scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    for epoch in range(_epochs):
        total_loss, steps, accum_loss = 0.0, 0, 0.0
        optimizer.zero_grad()
        for i in range(0, n, FINETUNE_BATCH):
            batch = ds[i : i + FINETUNE_BATCH]
            texts = get_texts(batch, text_col)
            enc   = {k: v.to(DEVICE) for k,v in tokenize(tok, texts, batch[label_col], is_regression).items()}
            with autocast(enabled=torch.cuda.is_available()):
                loss = model(**enc).loss / GRAD_ACCUM_STEPS
            SCALER.scale(loss).backward()
            accum_loss += loss.item()
            steps += 1
            if steps % GRAD_ACCUM_STEPS == 0:
                SCALER.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                SCALER.step(optimizer); SCALER.update()
                scheduler.step(); optimizer.zero_grad()
                total_loss += accum_loss; accum_loss = 0.0
        print(f'  {desc} epoch {epoch+1}/{_epochs}  loss={total_loss/max(steps//GRAD_ACCUM_STEPS,1):.4f}')
    model.eval()
    return model

print('Fine-tune function ready.')


In [ ]:
def _get_rot(block_size, device):
    if block_size in _ROT_PKL:
        R_base = _ROT_PKL[block_size].float().to(device)
        Q, _ = torch.linalg.qr(torch.randn(block_size-1, block_size-1, device=device))
        Q_full = torch.zeros(block_size, block_size, device=device)
        Q_full[0,0] = 1.0; Q_full[1:,1:] = Q
        return torch.matmul(R_base, Q_full)
    R = torch.eye(block_size, dtype=torch.float32, device=device)
    for i in range(block_size - 1):
        c = 1.0/math.sqrt(i+2); s = math.sqrt(i+1)/math.sqrt(i+2)
        rot_i = torch.eye(block_size, dtype=torch.float32, device=device)
        rot_i[i,i]=c; rot_i[i+1,i+1]=c; rot_i[i,i+1]=-s; rot_i[i+1,i]=s
        R = rot_i @ R
    R[[0,block_size-1]] = R[[block_size-1,0]]
    R[:,[0,block_size-1]] = R[:,[block_size-1,0]]
    return R

def _exch(R, i, j):
    R = R.clone(); R[[i,j]] = R[[j,i]]; R[:,[i,j]] = R[:,[j,i]]; return R

def _best_block_size(in_f):
    for d in range(1, in_f+1):
        if in_f % d == 0 and d >= 64: return min(d, BLOCK_SIZE_DEFAULT)
    for d in range(in_f, 0, -1):
        if in_f % d == 0 and d >= 16: return min(d, BLOCK_SIZE_DEFAULT)
    return max((d for d in range(1, in_f+1) if in_f % d == 0 and d <= BLOCK_SIZE_DEFAULT), default=1)

def _is_ffn(name):
    ffn = ['intermediate','output','ffn','fc','dense','up_proj','down_proj','gate_proj']
    att = ['attention','query','key','value','self','attn']
    gpt2_ffn = ['mlp.c_fc', 'mlp.c_proj']
    n = name.lower()
    is_bert_ffn = (any(k in n for k in ffn) and not any(k in n for k in att)
                   and 'classifier' not in n and 'pooler' not in n)
    is_gpt2_ffn = any(k in n for k in gpt2_ffn)
    return is_bert_ffn or is_gpt2_ffn

def rotation_duquant(W, block_size, max_steps):
    Wb = W.detach().clone().reshape(-1, block_size)
    Rot = _get_rot(block_size, Wb.device)
    exchange_ids, peak_vals = [], []
    for step in range(max_steps+1):
        r,c   = divmod(Wb.abs().argmax().item(), block_size)
        r2,c2 = divmod(Wb.abs().argmin().item(), block_size)
        peak_vals.append((Wb[r,c] - Wb[r2,c2]).item())
        if step == max_steps: break
        eid = r if Wb[r,c].abs() >= Wb[r2,c2].abs() else r2
        exchange_ids.append(eid)
        Wb = Wb @ _exch(Rot.clone(), 0, eid % block_size).to(Wb)
    best = int(torch.argmin(torch.tensor(peak_vals)).item())
    exchange_ids = exchange_ids[:best]
    R_ = torch.eye(block_size, dtype=W.dtype, device=W.device)
    for eid in exchange_ids:
        R_ = R_ @ _exch(Rot.clone(), 0, eid % block_size).to(R_)
    return (W.detach().clone().reshape(-1, block_size) @ R_).reshape(W.shape), exchange_ids, R_

def permutation_zigzag(W, block_size):
    W = W.detach().clone(); d = W.shape[-1]; nb = max(1, d//block_size)
    pairs = sorted([(i, W.abs().max(dim=0).values[i].item()) for i in range(d)],
                   key=lambda x: x[1], reverse=True)
    slots, cur, up = [[] for _ in range(nb)], 0, True
    for p in pairs:
        slots[cur].append(p)
        if up:
            cur += 1
            if cur == nb: cur -= 1; up = False
        else:
            cur -= 1
            if cur < 0: cur = 0; up = True
    for blk in slots: blk.sort(key=lambda x: x[1], reverse=True)
    perm = torch.zeros(d, dtype=torch.long)
    for bi, blk in enumerate(slots):
        for pos, (orig_col, _) in enumerate(blk):
            dst = bi*block_size + min(pos, block_size-1)
            if dst < d: perm[dst] = orig_col
    return W[:,perm], perm

def online_duquant_cali(W, block_size, act_scale=None, act_shift=None):
    if act_shift is not None: W = W - act_shift.unsqueeze(0).to(W.device)
    if act_scale is not None: W = W * act_scale.unsqueeze(0).clamp(min=1e-8).to(W.device)
    R_list, perm_list = [], []
    for _ in range(PERMUTATION_TIMES):
        W, _, R = rotation_duquant(W, block_size, MAX_ROTATION_STEP)
        R_list.append(R); W, perm = permutation_zigzag(W, block_size); perm_list.append(perm)
    W, _, R_final = rotation_duquant(W, block_size, MAX_ROTATION_STEP)
    R_list.append(R_final)
    return W, R_list, perm_list

print('DuQuant core ready.')


In [ ]:
def collect_act_scales_shifts(model, tok, ds_cfg, ds_name):
    path, config, train_split, _, text_col, label_col = ds_cfg
    ds = load_dataset(path, config, split=train_split) if config else load_dataset(path, split=train_split)
    ds = ds.shuffle(seed=0).select(range(min(N_CALIB_SAMPLES, len(ds))))
    scales, shifts = {}, {}
    hooks = []
    def make_hook(name):
        def hook(module, inp, out):
            x = inp[0].detach().float()
            if x.dim() == 3: x = x.view(-1, x.shape[-1])
            c_max  = x.abs().max(dim=0).values.cpu()
            c_min  = x.min(dim=0).values.cpu()
            c_max2 = x.max(dim=0).values.cpu()
            if name not in scales:
                scales[name] = c_max; shifts[name] = (c_max2 + c_min) / 2
            else:
                scales[name] = torch.maximum(scales[name], c_max)
                shifts[name] = 0.99*shifts[name] + 0.01*((c_max2 + c_min)/2)
        return hook
    model.to(DEVICE).eval()
    for name, mod in model.named_modules():
        if isinstance(mod, nn.Linear) and _is_ffn(name):
            hooks.append(mod.register_forward_hook(make_hook(name)))
    with torch.no_grad():
        for i in range(0, len(ds), BATCH_SIZE):
            batch = ds[i:i+BATCH_SIZE]
            enc = {k: v.to(DEVICE) for k,v in tokenize(tok, get_texts(batch, text_col)).items()}
            model(**enc)
    for h in hooks: h.remove()
    return scales, shifts

class LWCLinear(nn.Module):
    def __init__(self, linear):
        super().__init__()
        self.linear   = linear
        self.upbound  = nn.Parameter(torch.ones(linear.out_features, 1) * 4.0)
        self.lowbound = nn.Parameter(torch.ones(linear.out_features, 1) * 4.0)
        self.sigmoid  = nn.Sigmoid()
    def get_clipped_weight(self):
        W = self.linear.weight.data.float()
        xmax = W.abs().max(dim=1, keepdim=True).values
        xmax = self.sigmoid(self.upbound.to(W.device)) * xmax
        xmin = self.sigmoid(self.lowbound.to(W.device)) * (-xmax.abs())
        abs_max = torch.max(xmax.abs(), xmin.abs())
        scale   = (abs_max / 127.0).clamp(min=CLIPMIN, max=CLIPMAX)
        return (torch.clamp(torch.round(W / scale), -128, 127) * scale).to(self.linear.weight.dtype)
    def forward(self, x): return F.linear(x, self.get_clipped_weight(), self.linear.bias)

def apply_lwc(model):
    for name, module in list(model.named_modules()):
        if isinstance(module, nn.Linear) and _is_ffn(name):
            parts = name.split('.')
            parent = model
            for p in parts[:-1]: parent = getattr(parent, p)
            setattr(parent, parts[-1], LWCLinear(module))
    return model

def train_lwc(model, tok, ds_cfg, ds_name):
    path, config, train_split, _, text_col, label_col = ds_cfg
    is_regression = (ds_name == 'STSB')
    ds = load_dataset(path, config, split=train_split) if config else load_dataset(path, split=train_split)
    ds = ds.shuffle(seed=42).select(range(min(2000, len(ds))))
    lwc_params = [p for n,p in model.named_parameters() if 'bound' in n]
    if not lwc_params: return model
    optimizer = torch.optim.Adam(lwc_params, lr=LWC_LR)
    model.to(DEVICE).train()
    for epoch in range(LWC_EPOCHS):
        total_loss, steps = 0.0, 0
        for i in range(0, len(ds), FINETUNE_BATCH):
            batch = ds[i:i+FINETUNE_BATCH]
            enc = {k: v.to(DEVICE) for k,v in tokenize(tok, get_texts(batch, text_col), batch[label_col], is_regression).items()}
            optimizer.zero_grad()
            loss = model(**enc).loss
            loss.backward(); optimizer.step()
            total_loss += loss.item(); steps += 1
        print(f'    LWC epoch {epoch+1}/{LWC_EPOCHS}  loss={total_loss/max(steps,1):.4f}')
    model.eval()
    for name, module in list(model.named_modules()):
        if isinstance(module, LWCLinear):
            parts = name.split('.')
            parent = model
            for p in parts[:-1]: parent = getattr(parent, p)
            module.linear.weight.data = module.get_clipped_weight()
            setattr(parent, parts[-1], module.linear)
    return model

def apply_duquant(layer, name='', act_scale=None, act_shift=None):
    with torch.no_grad():
        W = layer.weight.data.float()
        bs = _best_block_size(W.shape[1])
        xmax = W.abs().max(dim=1, keepdim=True).values
        W = W.clamp(-SWC_RATIO * xmax, SWC_RATIO * xmax)
        W_t, R_list, perm_list = online_duquant_cali(W, bs, act_scale, act_shift)
        scaled_xmax = (LAC_RATIO * W_t.abs().max(dim=1, keepdim=True).values).clamp(min=CLIPMIN, max=CLIPMAX)                       if act_scale is not None else W_t.abs().max(dim=1, keepdim=True).values.clamp(min=CLIPMIN, max=CLIPMAX)
        scale = (scaled_xmax / 127.0).clamp(min=CLIPMIN)
        layer.weight.data = (torch.clamp(torch.round(W_t / scale), -128, 127) * scale).to(layer.weight.dtype)
        layer._dq_R    = R_list
        layer._dq_perm = perm_list
        layer._dq_bs   = bs
    return layer

def quantize_model(model, tok, ds_cfg, ds_name):
    print('  Collecting activation scales + shifts...')
    act_scales, act_shifts = collect_act_scales_shifts(model, tok, ds_cfg, ds_name)
    model.cpu()
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and _is_ffn(name):
            s = act_scales.get(name, None); sh = act_shifts.get(name, None)
            apply_duquant(module, name, s.cpu() if s is not None else None, sh.cpu() if sh is not None else None)
    print('  Running LWC optimization...')
    model = apply_lwc(model)
    model = train_lwc(model, tok, ds_cfg, ds_name)
    return model

print('Quantization pipeline ready.')


In [ ]:
def benchmark(model, tok, ds_cfg, ds_name):
    path, config, _, eval_split, text_col, label_col = ds_cfg
    is_regression = (ds_name == 'STSB')
    ds = load_dataset(path, config, split=eval_split) if config else load_dataset(path, split=eval_split)
    ds = ds.select(range(min(MAX_SAMPLES, len(ds))))

    model.to(DEVICE).eval()
    preds, labels_list, latencies, energies = [], [], [], []
    t_start = time.perf_counter()

    for i in range(0, len(ds), BATCH_SIZE):
        batch = ds[i : i + BATCH_SIZE]
        texts = get_texts(batch, text_col)
        enc   = {k: v.to(DEVICE) for k,v in tokenize(tok, texts).items()}
        sampler = PowerSampler(); sampler.start()
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model(**enc)
        if DEVICE == 'cuda': torch.cuda.synchronize()
        elapsed_ms   = (time.perf_counter() - t0) * 1000
        avg_power_mw = sampler.stop()
        n_items = len(batch[label_col])
        latencies.append(elapsed_ms / n_items)
        energies.append((avg_power_mw * elapsed_ms * 1e-3) / n_items)
        if is_regression:
            preds.extend(out.logits.squeeze(-1).cpu().tolist())
        else:
            preds.extend(out.logits.argmax(-1).cpu().tolist())
        labels_list.extend(batch[label_col])

    total_time = time.perf_counter() - t_start
    latency    = round(float(np.mean(latencies)), 4)
    throughput = round(len(ds) / total_time, 1)
    energy     = round(float(np.mean(energies)), 4)

    if is_regression:
        return {'Accuracy': float('nan'), 'F1': float('nan'),
                'MCC': float('nan'), 'Precision': float('nan'), 'Recall': float('nan'),
                'Pearson':  round(pearsonr(preds, labels_list)[0]  * 100, 2),
                'Latency_ms': latency, 'Throughput_sps': throughput, 'Energy_mJ': energy}

    acc  = round(accuracy_score(labels_list, preds) * 100, 2)
    f1   = round(f1_score(labels_list, preds, average='weighted', zero_division=0) * 100, 2)
    prec = round(precision_score(labels_list, preds, average='weighted', zero_division=0) * 100, 2)
    rec  = round(recall_score(labels_list, preds, average='weighted', zero_division=0) * 100, 2)
    mcc  = round(matthews_corrcoef(labels_list, preds) * 100, 2)
    return {'Accuracy': acc, 'F1': f1, 'MCC': mcc, 'Precision': prec, 'Recall': rec,
            'Pearson': float('nan'),
            'Latency_ms': latency, 'Throughput_sps': throughput, 'Energy_mJ': energy}

print('Benchmark function ready.')


In [ ]:
results = []

for ds_name, ds_cfg in DATASETS.items():
    for model_name, hf_id in MODELS.items():
        print(f'\n{"="*60}')
        print(f'  {ds_name} | {model_name}')
        print(f'{"="*60}')
        tok = pretrained_model = fp32_model = quantized_model = None
        try:
            n_labels = NUM_LABELS[ds_name]
            tok = AutoTokenizer.from_pretrained(hf_id)
            if model_name in GPT2_MODELS:
                tok = setup_gpt2_tokenizer(tok)
            elif tok.pad_token is None:
                tok.pad_token = tok.eos_token; tok.padding_side = 'left'

            pretrained_model = AutoModelForSequenceClassification.from_pretrained(
                hf_id, num_labels=n_labels, ignore_mismatched_sizes=True)
            if pretrained_model.config.pad_token_id is None:
                pretrained_model.config.pad_token_id = tok.eos_token_id
            if model_name in GPT2_MODELS:
                pretrained_model = patch_gpt2_conv1d(pretrained_model)

            # [1/6] Pretrained benchmark
            print('  [1/6] Benchmarking pretrained (no fine-tuning)...')
            pre_mem = memory_mb(pretrained_model)
            pre_m   = benchmark(pretrained_model, tok, ds_cfg, ds_name)
            results.append({'Dataset': ds_name, 'Model': model_name, 'Method': 'Pretrained', 'Bits': 32,
                            'Memory_MB': pre_mem, **pre_m})

            # [2/6] Fine-tune
            print('  [2/6] Fine-tuning (full dataset, 3 epochs, cosine LR, grad accum)...')
            fp32_model = pretrained_model; pretrained_model = None
            fp32_model = finetune(fp32_model, tok, ds_cfg, n_labels, ds_name)

            # [3/6] FP32 benchmark
            print('  [3/6] Benchmarking FP32 fine-tuned...')
            fp32_mem = memory_mb(fp32_model)
            fp32_m   = benchmark(fp32_model, tok, ds_cfg, ds_name)
            results.append({'Dataset': ds_name, 'Model': model_name, 'Method': 'FP32', 'Bits': 32,
                            'Memory_MB': fp32_mem, **fp32_m})

            # [4/6] Apply DuQuant
            print('  [4/6] Applying DuQuant (SWC + LAC + shift + LWC + FFN-only)...')
            quantized_model = quantize_model(copy.deepcopy(fp32_model), tok, ds_cfg, ds_name)

            # [5/6] QAT
            print('  [5/6] QAT (1 epoch @ LR*0.1)...')
            quantized_model = finetune(quantized_model, tok, ds_cfg, n_labels, ds_name,
                                       epochs=QAT_EPOCHS, lr=LR*0.1, desc='QAT')

            # [6/6] INT8 benchmark
            print('  [6/6] Benchmarking DuQuant INT8...')
            int8_mem = memory_mb(quantized_model)
            int8_m   = benchmark(quantized_model, tok, ds_cfg, ds_name)
            results.append({'Dataset': ds_name, 'Model': model_name, 'Method': 'DuQuant_INT8', 'Bits': 8,
                            'Memory_MB': int8_mem, **int8_m})
            print('  ✓ Done.')

        except Exception as e:
            import traceback
            print(f'  ERROR: {e}')
            traceback.print_exc()

        finally:
            for obj in [pretrained_model, fp32_model, quantized_model, tok]:
                try: del obj
                except: pass
            gc.collect()
            if DEVICE == 'cuda': torch.cuda.empty_cache()


In [ ]:
df = pd.DataFrame(results)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 320)
pd.set_option('display.float_format', '{:.2f}'.format)

for ds in df['Dataset'].unique():
    print(f'\n{"="*120}')
    print(f'  DATASET: {ds}')
    print(f'{"="*120}')
    sub = df[df['Dataset'] == ds]
    for method in ['Pretrained', 'FP32', 'DuQuant_INT8']:
        m = sub[sub['Method'] == method].drop(columns=['Dataset','Method']).set_index('Model')
        if not m.empty:
            print(f'\n  Method: {method}')
            print(m.to_string())
